# Day 28 - Data Cleaning and Pre-processing with Pandas

This notebook covers:
- Identifying missing values with `isna()` and `isnull()`
- Dropping missing records with `dropna()`
- Imputing missing values with mean, median, and mode
- Removing duplicate records with `drop_duplicates()`
- Renaming columns
- Fixing data types with `astype()` and `pd.to_numeric()`
- Cleaning text using the Pandas `.str` accessor
- Converting strings to datetime using `pd.to_datetime()`
- Extracting year, month, and day for time-series partitioning
- Data engineering style cleaning pipeline
- Interview questions


## 1. Import Libraries


In [ ]:
import pandas as pd
import numpy as np

print("pandas version:", pd.__version__)


pandas version: 2.2.2


## 2. Create a Messy E-commerce Dataset

The dataset below simulates common issues found in raw data:
- missing values
- duplicate records
- inconsistent column names
- numeric values stored as strings
- extra spaces in text
- inconsistent text casing
- invalid dates
- invalid amounts


In [ ]:
raw_orders = pd.DataFrame({
    "Order ID": [1001, 1002, 1003, 1004, 1005, 1005, 1006, 1007, 1008, 1009],
    "User_ID": [501, 502, np.nan, 504, 505, 505, 506, 507, 508, 509],
    "Customer Name": [" riya ", "AARAV", "kabir", "Meera", None, None, "isha ", " rohan", "Anaya", "Vikram"],
    "Amount": ["250.50", "-100", "500", "1,200.75", None, None, "750", "0", "999.99", "abc"],
    "Status": ["Completed", "completed", "Pending", " completed ", "Completed", "Completed", None, "Cancelled", "completed", "Completed"],
    "City": [" delhi", "Mumbai ", "BANGALORE", "Delhi", "Pune", "Pune", "chennai", None, "Mumbai", "Delhi"],
    "Order Date": ["2026-01-01", "2026/01/02", "01-03-2026", "invalid-date", "2026-01-05", "2026-01-05", None, "2026-01-07", "2026-01-08", "2026-01-09"],
    "Product Category": [" electronics ", "Electronics", "fashion", "Electronics", "Grocery", "Grocery", "Fashion", "electronics", None, "Grocery"]
})

raw_orders


,Order ID,User_ID,Customer Name,Amount,Status,City,Order Date,Product Category
0,1001,501.0,riya,250.50,Completed,delhi,2026-01-01,electronics
1,1002,502.0,AARAV,-100,completed,Mumbai,2026/01/02,Electronics
2,1003,NaN,kabir,500,Pending,BANGALORE,01-03-2026,fashion
3,1004,504.0,Meera,"1,200.75",completed,Delhi,invalid-date,Electronics
4,1005,505.0,None,None,Completed,Pune,2026-01-05,Grocery
5,1005,505.0,None,None,Completed,Pune,2026-01-05,Grocery
6,1006,506.0,isha,750,None,chennai,None,Fashion
7,1007,507.0,rohan,0,Cancelled,None,2026-01-07,electronics
8,1008,508.0,Anaya,999.99,completed,Mumbai,2026-01-08,None
9,1009,509.0,Vikram,abc,Completed,Delhi,2026-01-09,Grocery


## 3. Quick Data Inspection

Before cleaning a dataset, inspect:
- first few rows
- shape
- column names
- data types
- summary statistics
- missing values


In [ ]:
print("Shape:", raw_orders.shape)
print("Columns:", list(raw_orders.columns))
print("Data types:")
print(raw_orders.dtypes)


Shape: (10, 8)
Columns: ['Order ID', 'User_ID', 'Customer Name', 'Amount', 'Status', 'City', 'Order Date', 'Product Category']
Data types:
Order ID              int64
User_ID             float64
Customer Name        object
Amount               object
Status               object
City                 object
Order Date           object
Product Category     object
dtype: object


In [ ]:
raw_orders.head()


,Order ID,User_ID,Customer Name,Amount,Status,City,Order Date,Product Category
0,1001,501.0,riya,250.50,Completed,delhi,2026-01-01,electronics
1,1002,502.0,AARAV,-100,completed,Mumbai,2026/01/02,Electronics
2,1003,NaN,kabir,500,Pending,BANGALORE,01-03-2026,fashion
3,1004,504.0,Meera,"1,200.75",completed,Delhi,invalid-date,Electronics
4,1005,505.0,None,None,Completed,Pune,2026-01-05,Grocery


In [ ]:
raw_orders.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Order ID          10 non-null     int64  
 1   User_ID           9 non-null      float64
 2   Customer Name     8 non-null      object 
 3   Amount            8 non-null      object 
 4   Status            9 non-null      object 
 5   City              9 non-null      object 
 6   Order Date        9 non-null      object 
 7   Product Category  9 non-null      object 
dtypes: float64(1), int64(1), object(6)
memory usage: 772.0+ bytes


In [ ]:
raw_orders.describe(include="all")


,Order ID,User_ID,Customer Name,Amount,Status,City,Order Date,Product Category
count,10.000000,9.000000,8,8,9,9,9,9
unique,NaN,NaN,8,8,5,7,8,6
top,NaN,NaN,riya,250.50,Completed,Delhi,2026-01-05,Grocery
freq,NaN,NaN,1,1,4,2,2,3
mean,1005.000000,505.222222,NaN,NaN,NaN,NaN,NaN,NaN
std,2.581989,2.635231,NaN,NaN,NaN,NaN,NaN,NaN
min,1001.000000,501.000000,NaN,NaN,NaN,NaN,NaN,NaN
25%,1003.250000,504.000000,NaN,NaN,NaN,NaN,NaN,NaN
50%,1005.000000,505.000000,NaN,NaN,NaN,NaN,NaN,NaN
75%,1006.750000,507.000000,NaN,NaN,NaN,NaN,NaN,NaN


# Handling Missing Data


## 4. Identify Missing Values with `isna()`


In [ ]:
raw_orders.isna()


,Order ID,User_ID,Customer Name,Amount,Status,City,Order Date,Product Category
0,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False
2,False,True,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False
4,False,False,True,True,False,False,False,False
5,False,False,True,True,False,False,False,False
6,False,False,False,False,True,False,True,False
7,False,False,False,False,False,True,False,False
8,False,False,False,False,False,False,False,True
9,False,False,False,False,False,False,False,False


## 5. Count Missing Values per Column


In [ ]:
raw_orders.isna().sum()


,0
Order ID,0
User_ID,1
Customer Name,2
Amount,2
Status,1
City,1
Order Date,1
Product Category,1


## 6. `isnull()` Gives the Same Result as `isna()`


In [ ]:
raw_orders.isnull().sum()


,0
Order ID,0
User_ID,1
Customer Name,2
Amount,2
Status,1
City,1
Order Date,1
Product Category,1


## 7. Percentage of Missing Values


In [ ]:
missing_percentage = raw_orders.isna().mean() * 100
missing_percentage


,0
Order ID,0.0
User_ID,10.0
Customer Name,20.0
Amount,20.0
Status,10.0
City,10.0
Order Date,10.0
Product Category,10.0


## 8. Dropping Rows with Missing Values

`dropna()` removes rows containing missing values.

This should be used carefully because it may remove too much data.


In [ ]:
dropped_all_missing = raw_orders.dropna()

print("Original shape:", raw_orders.shape)
print("After dropna shape:", dropped_all_missing.shape)

dropped_all_missing


Original shape: (10, 8)
After dropna shape: (4, 8)


,Order ID,User_ID,Customer Name,Amount,Status,City,Order Date,Product Category
0,1001,501.0,riya,250.50,Completed,delhi,2026-01-01,electronics
1,1002,502.0,AARAV,-100,completed,Mumbai,2026/01/02,Electronics
3,1004,504.0,Meera,"1,200.75",completed,Delhi,invalid-date,Electronics
9,1009,509.0,Vikram,abc,Completed,Delhi,2026-01-09,Grocery


## 9. Drop Rows Based on Critical Columns Only

In data pipelines, some fields may be mandatory.

Example:
- order ID
- user ID
- amount

Rows missing these critical fields may be dropped.


In [ ]:
critical_clean = raw_orders.dropna(subset=["Order ID", "User_ID", "Amount"])

print("Original shape:", raw_orders.shape)
print("After critical drop shape:", critical_clean.shape)

critical_clean


Original shape: (10, 8)
After critical drop shape: (7, 8)


,Order ID,User_ID,Customer Name,Amount,Status,City,Order Date,Product Category
0,1001,501.0,riya,250.50,Completed,delhi,2026-01-01,electronics
1,1002,502.0,AARAV,-100,completed,Mumbai,2026/01/02,Electronics
3,1004,504.0,Meera,"1,200.75",completed,Delhi,invalid-date,Electronics
6,1006,506.0,isha,750,None,chennai,None,Fashion
7,1007,507.0,rohan,0,Cancelled,None,2026-01-07,electronics
8,1008,508.0,Anaya,999.99,completed,Mumbai,2026-01-08,None
9,1009,509.0,Vikram,abc,Completed,Delhi,2026-01-09,Grocery


## 10. Fill Missing Values with Default Values

`fillna()` can replace missing values with business defaults.


In [ ]:
filled_defaults = raw_orders.copy()

filled_defaults["Customer Name"] = filled_defaults["Customer Name"].fillna("Unknown")
filled_defaults["Status"] = filled_defaults["Status"].fillna("unknown")
filled_defaults["City"] = filled_defaults["City"].fillna("Unknown")
filled_defaults["Product Category"] = filled_defaults["Product Category"].fillna("Unknown")

filled_defaults


,Order ID,User_ID,Customer Name,Amount,Status,City,Order Date,Product Category
0,1001,501.0,riya,250.50,Completed,delhi,2026-01-01,electronics
1,1002,502.0,AARAV,-100,completed,Mumbai,2026/01/02,Electronics
2,1003,NaN,kabir,500,Pending,BANGALORE,01-03-2026,fashion
3,1004,504.0,Meera,"1,200.75",completed,Delhi,invalid-date,Electronics
4,1005,505.0,Unknown,None,Completed,Pune,2026-01-05,Grocery
5,1005,505.0,Unknown,None,Completed,Pune,2026-01-05,Grocery
6,1006,506.0,isha,750,unknown,chennai,None,Fashion
7,1007,507.0,rohan,0,Cancelled,Unknown,2026-01-07,electronics
8,1008,508.0,Anaya,999.99,completed,Mumbai,2026-01-08,Unknown
9,1009,509.0,Vikram,abc,Completed,Delhi,2026-01-09,Grocery


## 11. Impute Numeric Missing Values with Mean and Median

Mean and median are common imputation strategies for numeric columns.

Before imputing, numeric data should be converted into numeric dtype.


In [ ]:
orders_numeric = raw_orders.copy()

orders_numeric["Amount_clean"] = (
    orders_numeric["Amount"]
    .astype("string")
    .str.replace(",", "", regex=False)
)

orders_numeric["Amount_clean"] = pd.to_numeric(orders_numeric["Amount_clean"], errors="coerce")

print(orders_numeric[["Amount", "Amount_clean"]])
print("Mean amount:", orders_numeric["Amount_clean"].mean())
print("Median amount:", orders_numeric["Amount_clean"].median())


     Amount  Amount_clean
0    250.50         250.5
1      -100        -100.0
2       500         500.0
3  1,200.75       1200.75
4      None          <NA>
5      None          <NA>
6       750         750.0
7         0           0.0
8    999.99        999.99
9       abc          <NA>
Mean amount: 514.4628571428572
Median amount: 500.0


In [ ]:
mean_amount = orders_numeric["Amount_clean"].mean()
median_amount = orders_numeric["Amount_clean"].median()

orders_numeric["Amount_filled_mean"] = orders_numeric["Amount_clean"].fillna(mean_amount)
orders_numeric["Amount_filled_median"] = orders_numeric["Amount_clean"].fillna(median_amount)

orders_numeric[["Amount", "Amount_clean", "Amount_filled_mean", "Amount_filled_median"]]


,Amount,Amount_clean,Amount_filled_mean,Amount_filled_median
0,250.50,250.5,250.5,250.5
1,-100,-100.0,-100.0,-100.0
2,500,500.0,500.0,500.0
3,"1,200.75",1200.75,1200.75,1200.75
4,None,<NA>,514.462857,500.0
5,None,<NA>,514.462857,500.0
6,750,750.0,750.0,750.0
7,0,0.0,0.0,0.0
8,999.99,999.99,999.99,999.99
9,abc,<NA>,514.462857,500.0


## 12. Impute Categorical Missing Values with Mode

Mode is the most frequent value.

It is commonly used for categorical fields such as city, status, or category.


In [ ]:
city_mode = raw_orders["City"].mode()[0]
category_mode = raw_orders["Product Category"].mode()[0]

print("City mode:", city_mode)
print("Category mode:", category_mode)

orders_mode_filled = raw_orders.copy()
orders_mode_filled["City"] = orders_mode_filled["City"].fillna(city_mode)
orders_mode_filled["Product Category"] = orders_mode_filled["Product Category"].fillna(category_mode)

orders_mode_filled[["City", "Product Category"]]


City mode: Delhi
Category mode: Grocery


,City,Product Category
0,delhi,electronics
1,Mumbai,Electronics
2,BANGALORE,fashion
3,Delhi,Electronics
4,Pune,Grocery
5,Pune,Grocery
6,chennai,Fashion
7,Delhi,electronics
8,Mumbai,Grocery
9,Delhi,Grocery


# Data Quality Issues


## 13. Detect Duplicate Rows


In [ ]:
raw_orders.duplicated()


,0
0,False
1,False
2,False
3,False
4,False
5,True
6,False
7,False
8,False
9,False


## 14. Detect Duplicates Based on Specific Columns

A business key can define duplicates.

Example:
- `Order ID`
- `User_ID`


In [ ]:
raw_orders.duplicated(subset=["Order ID", "User_ID"])


,0
0,False
1,False
2,False
3,False
4,False
5,True
6,False
7,False
8,False
9,False


## 15. Remove Duplicate Records


In [ ]:
orders_no_duplicates = raw_orders.drop_duplicates(subset=["Order ID", "User_ID"], keep="first")

print("Original shape:", raw_orders.shape)
print("After duplicate removal:", orders_no_duplicates.shape)

orders_no_duplicates


Original shape: (10, 8)
After duplicate removal: (9, 8)


,Order ID,User_ID,Customer Name,Amount,Status,City,Order Date,Product Category
0,1001,501.0,riya,250.50,Completed,delhi,2026-01-01,electronics
1,1002,502.0,AARAV,-100,completed,Mumbai,2026/01/02,Electronics
2,1003,NaN,kabir,500,Pending,BANGALORE,01-03-2026,fashion
3,1004,504.0,Meera,"1,200.75",completed,Delhi,invalid-date,Electronics
4,1005,505.0,None,None,Completed,Pune,2026-01-05,Grocery
6,1006,506.0,isha,750,None,chennai,None,Fashion
7,1007,507.0,rohan,0,Cancelled,None,2026-01-07,electronics
8,1008,508.0,Anaya,999.99,completed,Mumbai,2026-01-08,None
9,1009,509.0,Vikram,abc,Completed,Delhi,2026-01-09,Grocery


## 16. Rename Columns

Raw files often contain spaces, mixed casing, or inconsistent naming.

Rename columns into a clean snake_case format.


In [ ]:
orders_renamed = orders_no_duplicates.rename(columns={
    "Order ID": "order_id",
    "User_ID": "user_id",
    "Customer Name": "customer_name",
    "Amount": "amount",
    "Status": "status",
    "City": "city",
    "Order Date": "order_date",
    "Product Category": "product_category"
})

orders_renamed.head()


,order_id,user_id,customer_name,amount,status,city,order_date,product_category
0,1001,501.0,riya,250.50,Completed,delhi,2026-01-01,electronics
1,1002,502.0,AARAV,-100,completed,Mumbai,2026/01/02,Electronics
2,1003,NaN,kabir,500,Pending,BANGALORE,01-03-2026,fashion
3,1004,504.0,Meera,"1,200.75",completed,Delhi,invalid-date,Electronics
4,1005,505.0,None,None,Completed,Pune,2026-01-05,Grocery


## 17. Standardize All Column Names Programmatically

This is useful when there are many columns.


In [ ]:
standardized_columns = raw_orders.copy()

standardized_columns.columns = (
    standardized_columns.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

standardized_columns.head()


,order_id,user_id,customer_name,amount,status,city,order_date,product_category
0,1001,501.0,riya,250.50,Completed,delhi,2026-01-01,electronics
1,1002,502.0,AARAV,-100,completed,Mumbai,2026/01/02,Electronics
2,1003,NaN,kabir,500,Pending,BANGALORE,01-03-2026,fashion
3,1004,504.0,Meera,"1,200.75",completed,Delhi,invalid-date,Electronics
4,1005,505.0,None,None,Completed,Pune,2026-01-05,Grocery


## 18. Fix Data Types with `astype()`

Use `astype()` when values are already clean and convertible.


In [ ]:
small_df = pd.DataFrame({
    "user_id": ["501", "502", "503"],
    "amount": ["100.50", "200.25", "300.00"]
})

small_df["user_id"] = small_df["user_id"].astype(int)
small_df["amount"] = small_df["amount"].astype(float)

print(small_df)
print(small_df.dtypes)


   user_id  amount
0      501  100.50
1      502  200.25
2      503  300.00
user_id      int64
amount     float64
dtype: object


## 19. Fix Dirty Numeric Columns with `pd.to_numeric()`

Use `pd.to_numeric(errors="coerce")` when values may be invalid.

Invalid values become NaN.


In [ ]:
orders_clean_types = orders_renamed.copy()

orders_clean_types["amount"] = (
    orders_clean_types["amount"]
    .astype("string")
    .str.replace(",", "", regex=False)
)

orders_clean_types["amount"] = pd.to_numeric(orders_clean_types["amount"], errors="coerce")
orders_clean_types["user_id"] = pd.to_numeric(orders_clean_types["user_id"], errors="coerce")

orders_clean_types[["order_id", "user_id", "amount"]]


,order_id,user_id,amount
0,1001,501.0,250.5
1,1002,502.0,-100.0
2,1003,NaN,500.0
3,1004,504.0,1200.75
4,1005,505.0,<NA>
6,1006,506.0,750.0
7,1007,507.0,0.0
8,1008,508.0,999.99
9,1009,509.0,<NA>


In [ ]:
orders_clean_types.dtypes


,0
order_id,int64
user_id,float64
customer_name,object
amount,Float64
status,object
city,object
order_date,object
product_category,object


# String Manipulation with Pandas `.str`


## 20. Clean Text Columns with `.str.strip()`

`str.strip()` removes leading and trailing spaces.


In [ ]:
text_cleaned = orders_clean_types.copy()

text_cleaned["customer_name"] = text_cleaned["customer_name"].astype("string").str.strip()
text_cleaned["status"] = text_cleaned["status"].astype("string").str.strip()
text_cleaned["city"] = text_cleaned["city"].astype("string").str.strip()
text_cleaned["product_category"] = text_cleaned["product_category"].astype("string").str.strip()

text_cleaned[["customer_name", "status", "city", "product_category"]]


,customer_name,status,city,product_category
0,riya,Completed,delhi,electronics
1,AARAV,completed,Mumbai,Electronics
2,kabir,Pending,BANGALORE,fashion
3,Meera,completed,Delhi,Electronics
4,<NA>,Completed,Pune,Grocery
6,isha,<NA>,chennai,Fashion
7,rohan,Cancelled,<NA>,electronics
8,Anaya,completed,Mumbai,<NA>
9,Vikram,Completed,Delhi,Grocery


## 21. Standardize Text Case

Common options:
- `.str.upper()`
- `.str.lower()`
- `.str.title()`


In [ ]:
text_cleaned["customer_name"] = text_cleaned["customer_name"].str.title()
text_cleaned["status"] = text_cleaned["status"].str.lower()
text_cleaned["city"] = text_cleaned["city"].str.title()
text_cleaned["product_category"] = text_cleaned["product_category"].str.title()

text_cleaned[["customer_name", "status", "city", "product_category"]]


,customer_name,status,city,product_category
0,Riya,completed,Delhi,Electronics
1,Aarav,completed,Mumbai,Electronics
2,Kabir,pending,Bangalore,Fashion
3,Meera,completed,Delhi,Electronics
4,<NA>,completed,Pune,Grocery
6,Isha,<NA>,Chennai,Fashion
7,Rohan,cancelled,<NA>,Electronics
8,Anaya,completed,Mumbai,<NA>
9,Vikram,completed,Delhi,Grocery


## 22. Replace Text Values

`str.replace()` can fix inconsistent values.


In [ ]:
category_fix = pd.DataFrame({
    "category": ["electronics ", "ELECTRONICS", "fashion", "grocery", "Home & Kitchen"]
})

category_fix["category_clean"] = (
    category_fix["category"]
    .str.strip()
    .str.lower()
    .str.replace("&", "and", regex=False)
    .str.title()
)

category_fix


,category,category_clean
0,electronics,Electronics
1,ELECTRONICS,Electronics
2,fashion,Fashion
3,grocery,Grocery
4,Home & Kitchen,Home And Kitchen


## 23. Filter Text with `.str.contains()`


In [ ]:
electronics_orders = text_cleaned[
    text_cleaned["product_category"].str.contains("Electronics", na=False)
]

electronics_orders


,order_id,user_id,customer_name,amount,status,city,order_date,product_category
0,1001,501.0,Riya,250.5,completed,Delhi,2026-01-01,Electronics
1,1002,502.0,Aarav,-100.0,completed,Mumbai,2026/01/02,Electronics
3,1004,504.0,Meera,1200.75,completed,Delhi,invalid-date,Electronics
7,1007,507.0,Rohan,0.0,cancelled,<NA>,2026-01-07,Electronics


# Handling Dates


## 24. Convert Strings to Datetime with `pd.to_datetime()`

Use `errors="coerce"` to convert invalid dates to NaT.


In [ ]:
date_cleaned = text_cleaned.copy()

date_cleaned["order_date"] = pd.to_datetime(date_cleaned["order_date"], errors="coerce")

date_cleaned[["order_id", "order_date"]]


,order_id,order_date
0,1001,2026-01-01
1,1002,NaT
2,1003,NaT
3,1004,NaT
4,1005,2026-01-05
6,1006,NaT
7,1007,2026-01-07
8,1008,2026-01-08
9,1009,2026-01-09


## 25. Identify Invalid Dates


In [ ]:
invalid_dates = date_cleaned[date_cleaned["order_date"].isna()]
invalid_dates[["order_id", "order_date"]]


,order_id,order_date
1,1002,NaT
2,1003,NaT
3,1004,NaT
6,1006,NaT


## 26. Extract Year, Month, and Day

Date parts are useful for time-series analysis and partitioning.


In [ ]:
date_cleaned["order_year"] = date_cleaned["order_date"].dt.year
date_cleaned["order_month"] = date_cleaned["order_date"].dt.month
date_cleaned["order_day"] = date_cleaned["order_date"].dt.day

date_cleaned[["order_id", "order_date", "order_year", "order_month", "order_day"]]


,order_id,order_date,order_year,order_month,order_day
0,1001,2026-01-01,2026.0,1.0,1.0
1,1002,NaT,NaN,NaN,NaN
2,1003,NaT,NaN,NaN,NaN
3,1004,NaT,NaN,NaN,NaN
4,1005,2026-01-05,2026.0,1.0,5.0
6,1006,NaT,NaN,NaN,NaN
7,1007,2026-01-07,2026.0,1.0,7.0
8,1008,2026-01-08,2026.0,1.0,8.0
9,1009,2026-01-09,2026.0,1.0,9.0


## 27. Date Partition Path Example

Data lakes often store data using partitions such as:

`year=2026/month=1/day=5`


In [ ]:
date_cleaned["partition_path"] = (
    "year=" + date_cleaned["order_year"].astype("Int64").astype("string") +
    "/month=" + date_cleaned["order_month"].astype("Int64").astype("string") +
    "/day=" + date_cleaned["order_day"].astype("Int64").astype("string")
)

date_cleaned[["order_id", "partition_path"]]


,order_id,partition_path
0,1001,year=2026/month=1/day=1
1,1002,<NA>
2,1003,<NA>
3,1004,<NA>
4,1005,year=2026/month=1/day=5
6,1006,<NA>
7,1007,year=2026/month=1/day=7
8,1008,year=2026/month=1/day=8
9,1009,year=2026/month=1/day=9


# Complete Data Cleaning Pipeline


## 28. Build a Clean Orders Dataset

Rules:
- remove duplicate order records
- rename columns
- clean strings
- convert amount to numeric
- convert date to datetime
- drop rows missing critical fields
- fill customer name and city defaults
- keep only amount greater than 0


In [ ]:
clean_orders = raw_orders.copy()

# Remove duplicates
clean_orders = clean_orders.drop_duplicates(subset=["Order ID", "User_ID"], keep="first")

# Rename columns
clean_orders = clean_orders.rename(columns={
    "Order ID": "order_id",
    "User_ID": "user_id",
    "Customer Name": "customer_name",
    "Amount": "amount",
    "Status": "status",
    "City": "city",
    "Order Date": "order_date",
    "Product Category": "product_category"
})

# Clean string columns
for column in ["customer_name", "status", "city", "product_category"]:
    clean_orders[column] = clean_orders[column].astype("string").str.strip()

clean_orders["customer_name"] = clean_orders["customer_name"].str.title()
clean_orders["status"] = clean_orders["status"].str.lower()
clean_orders["city"] = clean_orders["city"].str.title()
clean_orders["product_category"] = clean_orders["product_category"].str.title()

# Fix numeric data
clean_orders["amount"] = clean_orders["amount"].astype("string").str.replace(",", "", regex=False)
clean_orders["amount"] = pd.to_numeric(clean_orders["amount"], errors="coerce")
clean_orders["user_id"] = pd.to_numeric(clean_orders["user_id"], errors="coerce")

# Fix date data
clean_orders["order_date"] = pd.to_datetime(clean_orders["order_date"], errors="coerce")

# Fill non-critical missing text fields
clean_orders["customer_name"] = clean_orders["customer_name"].fillna("Unknown")
clean_orders["city"] = clean_orders["city"].fillna("Unknown")
clean_orders["product_category"] = clean_orders["product_category"].fillna("Unknown")
clean_orders["status"] = clean_orders["status"].fillna("unknown")

# Drop rows missing critical fields
clean_orders = clean_orders.dropna(subset=["order_id", "user_id", "amount", "order_date"])

# Keep only valid positive amounts
clean_orders = clean_orders[clean_orders["amount"] > 0]

# Add partition columns
clean_orders["order_year"] = clean_orders["order_date"].dt.year
clean_orders["order_month"] = clean_orders["order_date"].dt.month
clean_orders["order_day"] = clean_orders["order_date"].dt.day

clean_orders


,order_id,user_id,customer_name,amount,status,city,order_date,product_category,order_year,order_month,order_day
0,1001,501.0,Riya,250.5,completed,Delhi,2026-01-01,Electronics,2026,1,1
8,1008,508.0,Anaya,999.99,completed,Mumbai,2026-01-08,Unknown,2026,1,8


## 29. Validate the Cleaned Dataset


In [ ]:
print("Shape:", clean_orders.shape)
print("Missing values:")
print(clean_orders.isna().sum())
print("Data types:")
print(clean_orders.dtypes)


Shape: (2, 11)
Missing values:
order_id            0
user_id             0
customer_name       0
amount              0
status              0
city                0
order_date          0
product_category    0
order_year          0
order_month         0
order_day           0
dtype: int64
Data types:
order_id                     int64
user_id                    float64
customer_name       string[python]
amount                     Float64
status              string[python]
city                string[python]
order_date          datetime64[ns]
product_category    string[python]
order_year                   int32
order_month                  int32
order_day                    int32
dtype: object


## 30. Data Quality Summary Report


In [ ]:
quality_report = {
    "raw_records": len(raw_orders),
    "clean_records": len(clean_orders),
    "removed_records": len(raw_orders) - len(clean_orders),
    "duplicate_records_detected": raw_orders.duplicated(subset=["Order ID", "User_ID"]).sum(),
    "total_revenue": clean_orders["amount"].sum(),
    "average_order_value": clean_orders["amount"].mean(),
    "unique_users": clean_orders["user_id"].nunique()
}

quality_report


{'raw_records': 10,
 'clean_records': 2,
 'removed_records': 8,
 'duplicate_records_detected': np.int64(1),
 'total_revenue': np.float64(1250.49),
 'average_order_value': np.float64(625.245),
 'unique_users': 2}

## 31. Save Clean Output


In [ ]:
clean_orders.to_csv("clean_orders.csv", index=False)

try:
    clean_orders.to_parquet("clean_orders.parquet", index=False)
    print("Saved clean_orders.csv and clean_orders.parquet")
except Exception as error:
    print("Saved clean_orders.csv")
    print("Parquet save failed. Install pyarrow if needed.")
    print(error)


Saved clean_orders.csv and clean_orders.parquet


# Practice Problems


## 32. Practice Problem 1: Missing Value Audit

Create a missing value report for `raw_orders` showing:
- missing count per column
- missing percentage per column


In [ ]:
# Write solution here


## 33. Practice Problem 2: Drop vs Fill

Create two DataFrames:
1. One where rows with missing `User_ID` or `Amount` are dropped.
2. One where missing `Customer Name` is filled with `Unknown`.


In [ ]:
# Write solution here


## 34. Practice Problem 3: Remove Duplicates by User_ID

Remove duplicate records based on the `User_ID` column and keep the first record.


In [ ]:
# Write solution here


## 35. Practice Problem 4: Clean Text Columns

Clean `Customer Name`, `Status`, and `City`:
- remove spaces
- standardize casing


In [ ]:
# Write solution here


## 36. Practice Problem 5: Convert Dates and Create Partitions

Convert `Order Date` to datetime and create:
- year
- month
- day


In [ ]:
# Write solution here


## 37. Practice Problem 6: Build a Final Clean Dataset

Create a final clean dataset using these rules:
- remove duplicates
- convert amount to numeric
- convert dates to datetime
- keep only completed orders
- keep only amount greater than 0


In [ ]:
# Write solution here


# Interview Questions


## 38. How do you handle missing values in a dataset? When would you drop them vs fill them?

Common steps:
1. Identify missing values using `isna()` or `isnull()`.
2. Check missing value count and percentage.
3. Decide whether to drop or fill.

Drop missing values when:
- the missing field is critical
- very few rows are affected
- the row cannot be trusted without that field

Fill missing values when:
- the column is important
- missing values are common
- a reasonable replacement exists

Common fill strategies:
- mean for numeric columns
- median for skewed numeric columns
- mode for categorical columns
- default values such as `Unknown`


## 39. How do you change the data type of a column after loading a DataFrame?

Use `astype()` when the column is clean:

```python
df["user_id"] = df["user_id"].astype(int)
```

Use `pd.to_numeric()` when numeric values may contain invalid strings:

```python
df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
```

Use `pd.to_datetime()` for date columns:

```python
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
```


## 40. Code snippet to remove duplicate records based on a specific `User_ID` column

```python
df = df.drop_duplicates(subset=["User_ID"], keep="first")
```

To keep the latest duplicate instead:

```python
df = df.drop_duplicates(subset=["User_ID"], keep="last")
```


## 41. Summary

Key takeaways:
- use `isna()` or `isnull()` to identify missing values
- use `dropna()` to drop missing records
- use `fillna()` for imputation
- use `drop_duplicates()` to remove duplicate records
- use `rename()` or column string methods to clean column names
- use `astype()` for clean type conversions
- use `pd.to_numeric()` for dirty numeric columns
- use `.str` methods for text cleaning
- use `pd.to_datetime()` for date conversion
- extract year, month, and day for time-based partitioning
